# Exercise — Audit & Remediate Catalog Metadata (Customer & Marketing)

**Trailhead Provisions** is under a GDPR audit notice. Using the **AWS Glue Data Catalog**, find
where governance controls are missing for the two domains that hold personal data —
**Customer Identity & Loyalty** and **Marketing & Campaigns** — **fix the gaps in the catalog**,
verify with a re-audit, then write a short reconciliation memo. See `INSTRUCTIONS.md`.

## Step 0: Connect to your Cloud Lab
Run the cell below to configure your AWS Cloud Lab environment and seed the AWS Glue Data Catalog (`trailhead`) if it is not yet present in your lab account.

In [ ]:
import os
os.environ["CARDINAL_LF_LIVE"] = "1"
os.environ["CARDINAL_AWS_REGION"] = "us-east-1"
# If your Vocareum AWS credentials are not already in the environment, paste them below:
# os.environ["AWS_ACCESS_KEY_ID"] = "..."
# os.environ["AWS_SECRET_ACCESS_KEY"] = "..."
# os.environ["AWS_SESSION_TOKEN"] = "..."

import boto3
import setup_aws
# Seed the Glue Data Catalog and S3 tables if not yet present in the lab account
glue = boto3.client("glue", region_name=os.environ["CARDINAL_AWS_REGION"])
try:
    glue.get_database(Name="trailhead")
except glue.exceptions.EntityNotFoundException:
    setup_aws.main()

In [ ]:
import pandas as pd
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
from governance_toolkit import make_catalog, lf_backend

gc = make_catalog()
print(f"Backend: {lf_backend()} -> {type(gc).__name__} (Glue DB: {gc.database}, Region: {gc.region})")

## 1. Run the catalog audit (provided)

In [ ]:
audit = gc.catalog_audit()
print('Total gaps across all domains:', len(audit))
audit

## 2. Narrow to the Customer & Marketing domains
Filter the audit to the two domains you're responsible for, and inspect the column descriptions of `customer` and `campaign_membership` to see what each untagged column actually stores.

In [ ]:
# TODO(you): filter `audit` to the two domains you own and display the result.
my_domains = ["Customer Identity & Loyalty", "Marketing & Campaigns"]
my_gaps = None   # audit[audit["domain"].isin(...)]
my_gaps

In [ ]:
# Provided: inspect column descriptions so you can see what `customer.email` and `campaign_membership.cust_ref` store.
print("--- customer columns ---")
display(gc.describe("customer"))
print("--- campaign_membership columns ---")
display(gc.describe("campaign_membership"))

## 3. Remediate the gaps in the catalog
Write the missing governance metadata back to Glue with the provided helpers:
- `gc.set_table_metadata(table, owner=..., classification=..., retention=...)`
- `gc.tag_column(table, column, classification)`

Valid platform classification values are **`PII`**, **`SENSITIVE`**, and **`PUBLIC`**. Be sure to classify personal-data columns (and PII-bearing tables) as **`PII`**.

In [ ]:
# TODO(you): remediate EVERY gap in `my_gaps`, writing to the catalog.
#   gc.set_table_metadata(table, owner=..., classification=..., retention=...)
#   gc.tag_column(table, column, classification)   # valid classifications: "PII", "SENSITIVE", "PUBLIC"
print("TODO: write your remediation calls above")

## 4. Re-audit to verify (provided)
Confirm your two domains have 0 remaining gaps and that personal-data columns (`customer.email` and `campaign_membership.cust_ref`) carry the `PII` classification.

In [ ]:
re_audit = gc.catalog_audit()
remaining = re_audit[re_audit["domain"].isin(my_domains)]
email_cls = gc.catalog.table("customer").column("email").classification
cust_ref_cls = gc.catalog.table("campaign_membership").column("cust_ref").classification
print("Gaps in your domains -> before:", len(my_gaps) if my_gaps is not None else "?",
      "after:", len(remaining))
print(f"Column classifications -> customer.email={email_cls!r}, campaign_membership.cust_ref={cust_ref_cls!r}")
assert len(remaining) == 0, f"Expected 0 remaining gaps in {my_domains}, got {len(remaining)}"
assert email_cls == "PII" and cust_ref_cls == "PII", (
    "Both customer.email and campaign_membership.cust_ref hold personal data and must be tagged 'PII'"
)
remaining

## 5. Review the cross-domain conflicts (provided)

In [ ]:
gc.semantic_conflicts()

## 6. Write your reconciliation memo
Replace the cell below with your memo. Address every requirement in `INSTRUCTIONS.md`.

> **Your reconciliation memo here.** Address:
>
> - The untagged PII column(s) you found and fixed, and why each was the top GDPR risk.
> - The table-level gaps (missing owner / classification / retention) you remediated and what each meant.
> - The cross-domain semantic conflict(s) that involve these domains.
> - The tags/standards you recommend the platform enforce so the gaps don't recur.